# IRIS — YOLOv11s Screw Detection Training
> **Intelligent Robotic Identification and Sorting**  
> Model: `yolo11s.pt` | Epochs: 2000 | Early stop patience: 100

### Before running:
1. `Runtime → Change runtime type → T4 GPU` (free) or A100 (Colab Pro)
2. Fill in `CONFIG` cell below with your Roboflow API key and dataset info
3. Run all cells top to bottom

---
### Where to get the dataset
1. Go to [universe.roboflow.com](https://universe.roboflow.com)
2. Search **"screw detection"** — filter by Most Images, select a dataset with ≥1k images
3. Click **Download** → Format: `YOLOv11` → Show download code → copy the snippet
4. Paste your `api_key`, `workspace`, `project`, and `version` into the CONFIG cell

In [ ]:
# ============================================================
# CONFIG — edit these before running
# ============================================================
ROBOFLOW_API_KEY  = "YOUR_API_KEY_HERE"   # from app.roboflow.com → settings → API
ROBOFLOW_WORKSPACE = "YOUR_WORKSPACE"      # e.g. "jonah-chang"
ROBOFLOW_PROJECT   = "YOUR_PROJECT"        # e.g. "screw-detection"
ROBOFLOW_VERSION   = 1                     # dataset version number

MODEL_SIZE   = "yolo11s.pt"   # yolo11n (fastest) | yolo11s (recommended) | yolo11m
EPOCHS       = 2000
PATIENCE     = 100            # early stop if no improvement for N epochs
IMGSZ        = 640
BATCH        = 16             # reduce to 8 if OOM on free T4
PROJECT_NAME = "IRIS_screw_detection"
RUN_NAME     = "yolo11s_2000ep"
DEVICE       = 0              # GPU index; 'cpu' to force CPU

## 1. Environment Setup

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — check runtime type')

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 2. Dataset Download

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download("yolov11", location="/content/dataset")

DATASET_YAML = dataset.location + "/data.yaml"
print(f"Dataset location : {dataset.location}")
print(f"YAML             : {DATASET_YAML}")

In [ ]:
# Inspect dataset
import yaml
from pathlib import Path

with open(DATASET_YAML) as f:
    data = yaml.safe_load(f)

print("Classes  :", data.get('names'))
print("nc       :", data.get('nc'))

train_path = Path(dataset.location) / 'train' / 'images'
val_path   = Path(dataset.location) / 'valid' / 'images'
test_path  = Path(dataset.location) / 'test'  / 'images'

def count(p): return len(list(p.glob('*'))) if p.exists() else 0
print(f"Train: {count(train_path)} | Val: {count(val_path)} | Test: {count(test_path)}")

total = count(train_path) + count(val_path) + count(test_path)
if total < 1000:
    print(f"\n⚠  Only {total} images total — consider a larger dataset for best results")
else:
    print(f"\n✓  {total} total images — good to go")

## 3. Training

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_SIZE)

results = model.train(
    data      = DATASET_YAML,
    epochs    = EPOCHS,
    patience  = PATIENCE,
    imgsz     = IMGSZ,
    batch     = BATCH,
    device    = DEVICE,
    project   = PROJECT_NAME,
    name      = RUN_NAME,
    # Augmentation — tuned for small industrial objects
    hsv_h     = 0.015,   # hue shift (minimal — screws don't change hue)
    hsv_s     = 0.5,     # saturation
    hsv_v     = 0.4,     # brightness
    degrees   = 15.0,    # rotation — screws appear at any angle
    translate = 0.1,
    scale     = 0.5,
    shear     = 2.0,
    flipud    = 0.5,     # screws look same upside-down
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.1,
    copy_paste= 0.1,     # paste extra screw instances into scenes
    # Optimizer
    optimizer = 'AdamW',
    lr0       = 0.001,
    lrf       = 0.01,
    warmup_epochs = 5,
    # Regularization
    weight_decay = 0.0005,
    dropout   = 0.0,
    # Logging
    plots     = True,
    save      = True,
    save_period = 50,    # checkpoint every 50 epochs
    verbose   = True,
)

## 4. Validation

In [ ]:
best_weights = f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt"
model_best = YOLO(best_weights)

metrics = model_best.val(
    data   = DATASET_YAML,
    imgsz  = IMGSZ,
    device = DEVICE,
    split  = 'test',
)

print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

if metrics.box.map50 < 0.85:
    print("\n⚠  mAP@0.5 below 0.85 target — consider more data or longer training")
else:
    print("\n✓  mAP@0.5 target met")

## 5. Export

In [ ]:
# Export to ONNX (cross-platform, used by host PC inference)
model_best.export(
    format   = 'onnx',
    imgsz    = IMGSZ,
    opset    = 17,
    simplify = True,
    dynamic  = False,   # fixed batch=1 for real-time inference
)
print("ONNX export complete")

In [ ]:
# Quick inference smoke-test on one val image
from pathlib import Path
import random

val_images = list((Path(dataset.location) / 'valid' / 'images').glob('*'))
if val_images:
    sample = random.choice(val_images)
    preds = model_best(str(sample), conf=0.25, iou=0.45)
    preds[0].show()   # inline display in Colab
    print(f"Detections on {sample.name}: {len(preds[0].boxes)} screws")

## 6. Download Weights

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

run_dir  = Path(f"{PROJECT_NAME}/{RUN_NAME}")
best_pt  = run_dir / 'weights' / 'best.pt'
best_onnx = best_pt.with_suffix('.onnx')

# Bundle best.pt + best.onnx into a zip
shutil.copy(best_pt,   '/content/IRIS_best.pt')
if best_onnx.exists():
    shutil.copy(best_onnx, '/content/IRIS_best.onnx')

# Also zip the full run (metrics, curves, confusion matrix)
shutil.make_archive('/content/IRIS_run', 'zip', str(run_dir))

print("Downloading weights...")
files.download('/content/IRIS_best.pt')
if Path('/content/IRIS_best.onnx').exists():
    files.download('/content/IRIS_best.onnx')
files.download('/content/IRIS_run.zip')
print("Done — place best.pt and best.onnx in IRIS Progect/models/")

---
## Appendix — Augmented Dataset Builder (Optional)
> Run this section **only** if your downloaded dataset has fewer than 1000 images.  
> It uses Albumentations to synthetically expand the dataset 3x via heavy augmentation.

In [ ]:
# OPTIONAL: run only if dataset < 1000 images
!pip install -q albumentations

import albumentations as A
import cv2
import numpy as np
from pathlib import Path

aug_pipeline = A.Compose([
    A.RandomBrightnessContrast(p=0.6),
    A.GaussNoise(var_limit=(10, 50), p=0.4),
    A.MotionBlur(blur_limit=5, p=0.3),
    A.HueSaturationValue(p=0.4),
    A.RandomGamma(p=0.3),
    A.CLAHE(p=0.3),
    A.Perspective(scale=(0.02, 0.05), p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def augment_split(split='train', multiplier=3):
    img_dir  = Path(dataset.location) / split / 'images'
    lbl_dir  = Path(dataset.location) / split / 'labels'
    added = 0
    for img_path in list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            continue
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        bboxes, classes = [], []
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            classes.append(int(parts[0]))
            bboxes.append([float(x) for x in parts[1:5]])
        for i in range(multiplier - 1):
            try:
                out = aug_pipeline(image=img, bboxes=bboxes, class_labels=classes)
            except Exception:
                continue
            stem = f"{img_path.stem}_aug{i}"
            cv2.imwrite(str(img_dir / f"{stem}.jpg"),
                        cv2.cvtColor(out['image'], cv2.COLOR_RGB2BGR))
            lines = [f"{c} {' '.join(f'{v:.6f}' for v in b)}"
                     for c, b in zip(out['class_labels'], out['bboxes'])]
            (lbl_dir / f"{stem}.txt").write_text('\n'.join(lines))
            added += 1
    print(f"{split}: added {added} augmented images")

augment_split('train', multiplier=3)
augment_split('valid', multiplier=2)

# Re-count
new_total = sum(count(Path(dataset.location) / s / 'images') for s in ['train','valid','test'])
print(f"Total after augmentation: {new_total} images")